# Steerable Pyramid

Used https://pyrtools.readthedocs.io/en/latest/ for steerabe pyramid. not my own converted matlab functions

**Core functions**

In [ ]:
import numpy as np
from pyrtools.pyramids import SteerablePyramidFreq


def norm_energies(pyr_coeffs, num_orientations, sigma=0.1):
    """
    Compute contrast-normalized local energy responses from a steerable pyramid.

    Parameters:
        pyr_coeffs (dict): Coefficients of the steerable pyramid.
        num_orientations (int): Number of orientation subbands at each scale.
        sigma (float): Semi-saturation constant for contrast normalization.

    Returns:
        dict: Normalized coefficients.
    """
    # Compute energy (magnitude squared)
    energies = {k: np.abs(v) ** 2 for k, v in pyr_coeffs.items()}

    # Initialize normalizers
    normalizers = {k: np.zeros_like(v) for k, v in pyr_coeffs.items()}

    # Add high-pass normalizer
    if "residual_highpass" in pyr_coeffs:
        normalizers["residual_highpass"] = energies["residual_highpass"]

    # Compute normalizers across scales and orientations
    num_levels = (len(pyr_coeffs) - 2) // num_orientations  # Exclude high/low residuals
    for level in range(num_levels):
        for orientation in range(num_orientations):
            key = (level, orientation)
            if key in pyr_coeffs:
                normalizers[key] += energies[key]

    # Normalize each bandpass energy band
    normalized_coeffs = {}
    for level in range(num_levels):
        for orientation in range(num_orientations):
            key = (level, orientation)
            if key in pyr_coeffs:
                norm_value = (
                    normalizers.get((level - 1, orientation), 0)
                    + normalizers.get((level, orientation), 0)
                    + normalizers.get((level + 1, orientation), 0)
                ) + sigma**2  # Prevent divide by zero

                normalized_coeffs[key] = energies[key] / norm_value

    # Add low-pass and high-pass residuals without modification
    if "residual_highpass" in pyr_coeffs:
        normalized_coeffs["residual_highpass"] = pyr_coeffs["residual_highpass"]

    if "residual_lowpass" in pyr_coeffs:
        normalized_coeffs["residual_lowpass"] = pyr_coeffs["residual_lowpass"]

    return normalized_coeffs

In [ ]:
import numpy as np
import cv2
import h5py
import pyrtools as pt
import os
import scipy.io as sio
import matplotlib.pyplot as plt
import json
import time


# File path (Modify according to your Google Drive structure)
hdf5_file = "G:/.shortcut-targets-by-id/1iPft89M8JBdXa7g3K5MBL0oLkoXgu5HL/V2_drift/nsd_raw/nsddata_stimuli/stimuli/nsd/nsd_stimuli.hdf5"


class NSDStimulusProcessor:
    def height(self, dims, bandwidth, allow_high_levels=False):
        """Calculate the number of spatial frequency levels for the steerable pyramid."""
        max_ht = int(np.log2(min(dims)))  # Compute max possible levels
        num_levels = int(np.log2(min(dims)) - np.log2(bandwidth))  # Default calculation

        if allow_high_levels:
            return min(num_levels, max_ht)  # Allow as many levels as possible
        return min(num_levels, 7)  # Default to 7 if allow_high_levels=False

    def __init__(
        self,
        stim_filename,
        output_folder="pyramid_expand/",
        max_images=5,
        allow_high_levels=False,
    ):
        """Initialize the processor with required parameters."""
        self.stim_filename = stim_filename
        self.output_folder = output_folder
        self.max_images = max_images
        self.allow_high_levels = allow_high_levels
        self.interp_img_size = 724
        self.background_size = 1024
        self.img_scaling = 0.5
        self.num_orientations = 8
        self.bandwidth = 1
        self.dims = [512, 512]  # Final downsampled dimensions
        os.makedirs(output_folder, exist_ok=True)

        # Open HDF5 file and get dataset shape **without loading images**
        with h5py.File(self.stim_filename, "r") as f:
            dataset_shape = f["imgBrick"].shape  # (73000, 425, 425, 3)
            self.total_images = dataset_shape[0]
            self.img_size_x, self.img_size_y, _ = dataset_shape[1:]

        self.num_imgs = min(self.max_images, self.total_images)
        print(
            f"Dataset contains {self.total_images} images. Loading only {self.num_imgs}."
        )

    def load_image(self, img_num):
        """Load a single image from HDF5 file on demand to save memory."""
        if img_num >= self.total_images:
            raise IndexError(
                f"Image index {img_num} is out of range (0-{self.total_images-1})."
            )

        with h5py.File(self.stim_filename, "r") as f:
            orig_img = f["imgBrick"][img_num]  # Load only one image at a time

        orig_img = np.array(orig_img, dtype=np.float32)  # Convert to float
        return orig_img  # Shape: (425, 425, 3)

    def interpolate_image(self, img):
        """Rescale image to `interp_img_size` while preserving details."""
        xq, yq = np.meshgrid(
            np.linspace(0, self.img_size_x - 1, self.interp_img_size),
            np.linspace(0, self.img_size_y - 1, self.interp_img_size),
        )
        interp_img = np.zeros((self.interp_img_size, self.interp_img_size, 3))
        for i in range(3):
            interp_img[:, :, i] = cv2.remap(
                img[:, :, i],
                xq.astype(np.float32),
                yq.astype(np.float32),
                cv2.INTER_LINEAR,
            )
        return interp_img

    def add_fixation_point(self, img):
        """Add a semi-transparent red fixation point at the center."""
        fixation_radius = 8
        fixation_color = (255, 0, 0)  # Red
        overlay = img.copy()
        cv2.circle(
            overlay,
            (self.interp_img_size // 2, self.interp_img_size // 2),
            fixation_radius,
            fixation_color,
            -1,
        )
        return cv2.addWeighted(overlay, 0.5, img, 0.5, 0)

    def pad_background(self, img):
        """Pad image to `background_size` with gray background."""
        big_img = np.full(
            (self.background_size, self.background_size, 3), 127, dtype=np.uint8
        )
        pad_start = (self.background_size - self.interp_img_size) // 2
        big_img[
            pad_start : pad_start + self.interp_img_size,
            pad_start : pad_start + self.interp_img_size,
            :,
        ] = img
        return big_img

    def apply_steerable_pyramid(self, img):
        """Apply steerable pyramid transformation after resizing."""
        img_shape = img.shape[:2]  # Get final resized image dimensions
        self.num_levels = self.height(
            img_shape, self.bandwidth, self.allow_high_levels
        )  # Store levels

        print(
            f"Applying Steerable Pyramid with {self.num_levels} levels for {img_shape} image size."
        )

        # Convert image to float32 and normalize between -1 and 1
        img = img.astype(np.float32)
        img = (img - np.mean(img)) / np.std(img)  # Standardize image

        # Ensure grayscale shape (H, W)
        if len(img.shape) == 3:
            img = np.mean(img, axis=2)  # Convert to grayscale if necessary
            print(f"Converted to grayscale, new shape: {img.shape}")

        # Ensure input shape matches what the pyramid expects
        if img.shape != (512, 512):
            print(f"Warning: Unexpected image shape {img.shape} before pyramid.")

        return pt.pyramids.SteerablePyramidFreq(
            img, height=self.num_levels, order=self.num_orientations
        )

    def process_image(self, img_num, norm_resp=False):
        """Full pipeline: load, process, apply pyramid filters, and save."""
        start_time = time.time()  # Track processing time

        # MATLAB-style 1-based indexing for filenames
        pyramid_filename = f"pyrImg{img_num}.mat"
        save_path = os.path.join(self.output_folder, pyramid_filename)
        if os.path.exists(save_path):
            print(f"Image {img_num} already processed. Skipping...")
            return  # Skip processing

        print(f"Processing Image {img_num}/{self.num_imgs}")

        try:
            # Load and preprocess image
            img = self.load_image(img_num)
            img = self.interpolate_image(img)
            img = self.add_fixation_point(img)
            img = self.pad_background(img)

            img_gray = np.mean(img, axis=2).astype(np.float32)  # Convert to grayscale
            img_gray = (img_gray - np.mean(img_gray)) / np.std(img_gray)  # Normalize

            # Downsample
            img_resized = cv2.resize(
                img_gray, (self.dims[1], self.dims[0]), interpolation=cv2.INTER_AREA
            )

            # Apply Steerable Pyramid
            print(f"Applying Steerable Pyramid to Image {img_num}")
            pyr = self.apply_steerable_pyramid(img_resized)
            print(f"Steerable Pyramid successfully applied to Image {img_num}")

            # Apply Contrast Normalization (if enabled)
            if norm_resp:
                pyr_coeffs = norm_energies(
                    pyr.pyr_coeffs, self.num_orientations, sigma=0.1
                )
                print("Applied normalized responses.")
            else:
                pyr_coeffs = pyr.pyr_coeffs

            print(f"Extracting frequency responses for Image {img_num}")
            sum_ori = [
                np.zeros_like(img_resized, dtype=np.float32)
                for _ in range(self.num_levels + 2)
            ]
            model_ori = [
                np.zeros((self.num_orientations, *img_resized.shape), dtype=np.float32)
                for _ in range(self.num_levels + 2)
            ]

            print(
                f"Pyramid has {len(pyr_coeffs)} coefficients: {list(pyr_coeffs.keys())}"
            )

            # Process Each Level and Orientation
            for ilev in range(self.num_levels):
                for orientation in range(self.num_orientations):
                    key = (ilev, orientation)
                    if key in pyr_coeffs:
                        this_band = np.abs(pyr_coeffs[key])
                        this_band_resized = cv2.resize(
                            this_band,
                            (self.dims[1], self.dims[0]),
                            interpolation=cv2.INTER_LINEAR,
                        )

                        sum_ori[ilev] += this_band_resized
                        model_ori[ilev][orientation] = this_band_resized
                    else:
                        print(f"Warning: Missing pyramid key {key}")

            # Remove extra orientations (ignore 8th orientation)
            # Extract only valid coefficients: 7 levels × 8 orientations
            pyr_coeffs = {
                k: v
                for k, v in pyr.pyr_coeffs.items()
                if (
                    isinstance(k, tuple) and k[1] < 8
                )  # Keep only 8 orientations per level
                or k in ["residual_highpass", "residual_lowpass"]  # Keep low/high pass
            }
            # Add Low-Frequency Band
            if "residual_lowpass" in pyr_coeffs:
                lowpass = np.abs(pyr_coeffs["residual_lowpass"])
                lowpass_resized = cv2.resize(
                    lowpass,
                    (self.dims[1], self.dims[0]),
                    interpolation=cv2.INTER_LINEAR,
                )
                sum_ori[self.num_levels] = lowpass_resized
                model_ori[self.num_levels] = lowpass_resized
            else:
                print("Warning: Low-pass residual missing.")

            # Add High-Frequency Band
            if "residual_highpass" in pyr_coeffs:
                highpass = np.abs(pyr_coeffs["residual_highpass"])
                highpass_resized = cv2.resize(
                    highpass,
                    (self.dims[1], self.dims[0]),
                    interpolation=cv2.INTER_LINEAR,
                )
                sum_ori[self.num_levels + 1] = highpass_resized
                model_ori[self.num_levels + 1] = highpass_resized
            else:
                print("Warning: High-pass residual missing.")

            print(f"Extracting frequency responses for Image {img_num}")
            print(
                f"Pyramid has {len(pyr_coeffs)} coefficients: {list(pyr_coeffs.keys())}"
            )

            # Save to .mat file
            sio.savemat(
                save_path,
                {
                    "interpImgSize": np.array(self.interp_img_size, dtype=np.int32),
                    "backgroundSize": np.array(self.background_size, dtype=np.int32),
                    "imgScaling": np.array(self.img_scaling, dtype=np.float32),
                    "numOrientations": np.array(self.num_orientations, dtype=np.int32),
                    "bandwidth": np.array(self.bandwidth, dtype=np.int32),
                    "dims": np.array(self.dims, dtype=np.int32),
                    "bigImg": img_gray.astype(np.float32),  # Ensure correct dtype
                    "sumOri": [s.astype(np.float32) for s in sum_ori],  # Keep as a list
                    "modelOri": [
                        m.astype(np.float32) for m in model_ori
                    ],  # Keep as a list
                    "numLevels": np.array(self.num_levels, dtype=np.int32),
                    "normResp": np.array(
                        norm_resp, dtype=np.int8
                    ),  # Save normalization status
                },
            )

            print(
                f"Saved: {save_path} (Time taken: {time.time() - start_time:.2f} sec)"
            )

        except Exception as e:
            print(f"ERROR processing image {img_num}: {e}")

    def visualize_pyramid(self, img_num):
        """Visualize the original image, preprocessed image, and steerable pyramid decomposition."""

        # Check if img_num is within the loaded subset
        if img_num >= self.num_imgs:
            print(f"Error: Image index {img_num} is out of range (0-{self.num_imgs-1})")
            return

        print(f"Visualizing Steerable Pyramid for Image {img_num}")

        # Load the original image
        img = self.load_image(img_num)

        # Preprocess the image (interpolation, grayscale, etc.)
        img_interpolated = self.interpolate_image(img)
        img_with_fixation = self.add_fixation_point(img_interpolated)
        img_with_padding = self.pad_background(img_with_fixation)
        img_gray = np.mean(img_with_padding, axis=2)

        # Step 5: Downsample for steerable pyramid
        img_resized = cv2.resize(
            img_gray, (self.dims[1], self.dims[0]), interpolation=cv2.INTER_AREA
        )

        # Apply the Steerable Pyramid
        pyr = self.apply_steerable_pyramid(img_resized)

        # Visualization
        fig = plt.figure(figsize=(12, 7))

        # Visualize the original image
        ax1 = fig.add_subplot(2, 1, 1)
        ax1.imshow(img / 255.0)  # Normalize to [0, 1] for display
        ax1.set_title("Original Image")
        ax1.axis("off")

        # Visualize the preprocessed image
        ax2 = fig.add_subplot(2, 1, 2)
        ax2.imshow(img_resized, cmap="gray")
        ax2.set_title("Preprocessed Image")
        ax2.axis("off")

        plt.tight_layout()
        plt.show()

        # Subplots for Steerable Pyramid
        num_levels = pyr.num_scales
        num_orientations = self.num_orientations

        # Subplot for regular pyramid filters
        fig_filters, axes_filters = plt.subplots(
            num_levels, num_orientations, figsize=((20, 15))
        )
        fig_filters.suptitle(
            f"Steerable Pyramid - Regular Filters (Image {img_num})", fontsize=20
        )

        for ilev in range(num_levels):
            for orientation in range(num_orientations):
                key = (ilev, orientation)
                ax = axes_filters[ilev, orientation]
                if key in pyr.pyr_coeffs:
                    band = np.abs(
                        pyr.pyr_coeffs[key]
                    )  # Take absolute value for display
                    ax.imshow(band, cmap="gray")
                    ax.set_title(f"Level {ilev+1}, Ori {orientation+1}", fontsize=8)
                else:
                    ax.axis("off")
                ax.set_xticks([])
                ax.set_yticks([])

        plt.tight_layout()
        plt.show()

        # Subplot for high-pass and low-pass residuals
        fig_residuals, axes_residuals = plt.subplots(1, 2, figsize=(15, 5))
        fig_residuals.suptitle(
            f"Steerable Pyramid - High/Low-Pass Residuals (Image {img_num})",
            fontsize=20,
        )

        # High-pass residual
        if "residual_highpass" in pyr.pyr_coeffs:
            ax_hp = axes_residuals[0]
            ax_hp.imshow(np.abs(pyr.pyr_coeffs["residual_highpass"]), cmap="gray")
            ax_hp.set_title("High-Pass Residual", fontsize=14)
            ax_hp.axis("off")

        # Low-pass residual
        if "residual_lowpass" in pyr.pyr_coeffs:
            ax_lp = axes_residuals[1]
            ax_lp.imshow(np.abs(pyr.pyr_coeffs["residual_lowpass"]), cmap="gray")
            ax_lp.set_title("Low-Pass Residual", fontsize=14)
            ax_lp.axis("off")

        # Adjust layout
        plt.tight_layout()
        plt.show()

    def visualize_pyramid_with_fixation(self, img_num):
        """Visualize the original image, preprocessed image, and zoom in on the fixation point."""

        # Check if img_num is within the loaded subset
        if img_num >= self.num_imgs:
            print(f"Error: Image index {img_num} is out of range (0-{self.num_imgs-1})")
            return

        print(f"Visualizing Preprocessing Steps and Fixation Point for Image {img_num}")

        # Load and preprocess the image
        img = self.load_image(img_num)
        img_interpolated = self.interpolate_image(img)
        img_with_fixation = self.add_fixation_point(img_interpolated)
        img_with_padding = self.pad_background(img_with_fixation)

        # Visualization
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle(
            f"Preprocessing Steps for Image {img_num} and Fixation Point", fontsize=16
        )

        # Full preprocessed image with fixation
        axes[0].imshow(img_with_padding / 255.0)  # Normalize to [0, 1]
        axes[0].set_title("Preprocessed Image with Fixation")
        axes[0].axis("off")

        # Zoom into the fixation area
        fixation_radius = 20  # Define the region to zoom in (e.g., 20x20 pixels)
        center = self.background_size // 2
        zoom_img = img_with_padding[
            center - fixation_radius : center + fixation_radius,
            center - fixation_radius : center + fixation_radius,
            :,
        ]
        axes[1].imshow(zoom_img / 255.0)  # Normalize for display
        axes[1].set_title("Zoomed-in Fixation Area")
        axes[1].axis("off")

        # Show the red channel only
        red_channel = zoom_img[:, :, 0]
        axes[2].imshow(
            red_channel, cmap="Reds"
        )  # Use "Reds" colormap to highlight red intensity
        axes[2].set_title("Red Channel of Fixation Area")
        axes[2].axis("off")

        plt.tight_layout()
        plt.show()

    def process_batch(self, start_idx, end_idx):
        """Process a batch of images while tracking progress."""
        batch_start_time = time.time()
        log_file = os.path.join(self.output_folder, "progress_log.json")

        # Ensure the log file is created if it doesn't exist
        if not os.path.exists(log_file):
            print("Creating new progress log file.")
            progress_data = {}
            with open(log_file, "w") as f:
                json.dump(progress_data, f, indent=4)
        else:
            try:
                with open(log_file, "r") as f:
                    progress_data = json.load(f)
            except json.JSONDecodeError:
                print("Warning: Log file corrupted. Resetting progress tracking.")
                progress_data = {}
                with open(log_file, "w") as f:
                    json.dump(progress_data, f, indent=4)

        print(f"Processing batch from {start_idx} to {end_idx - 1}")

        for img_num in range(start_idx, end_idx):
            try:
                pyramid_filename = f"pyrImg{img_num}.mat"
                save_path = os.path.join(self.output_folder, pyramid_filename)
                if str(img_num) in progress_data and os.path.exists(save_path):
                    print(f"Skipping Image {img_num} (already processed)")
                    continue

                self.process_image(img_num)
                progress_data[str(img_num)] = "processed"

                # Save the log file after each processed image
                with open(log_file, "w") as f:
                    json.dump(progress_data, f, indent=4)

            except Exception as e:
                print(f"Error processing image {img_num}: {e}. Skipping...")
                continue

        print(
            f"Batch {start_idx} - {end_idx} completed in {time.time() - batch_start_time:.2f} sec."
        )

    def process_all_in_batches(self, batch_size=10000):
        """Process all images in batches while skipping already processed images."""
        total_start_time = time.time()
        log_file = os.path.join(self.output_folder, "progress_log.json")

        # Ensure log file exists before processing
        if not os.path.exists(log_file):
            print("Creating new progress log file.")
            progress_data = {}
            with open(log_file, "w") as f:
                json.dump(progress_data, f, indent=4)
        else:
            try:
                with open(log_file, "r") as f:
                    progress_data = json.load(f)
            except json.JSONDecodeError:
                print("Warning: Log file corrupted. Resetting progress tracking.")
                progress_data = {}
                with open(log_file, "w") as f:
                    json.dump(progress_data, f, indent=4)

        total_batches = (self.total_images + batch_size - 1) // batch_size
        print(
            f"Total images: {self.total_images}. Processing in {total_batches} batches of {batch_size} images."
        )

        for batch_idx in range(total_batches):
            start_idx = batch_idx * batch_size
            end_idx = min((batch_idx + 1) * batch_size, self.total_images)

            # Skip fully processed batches
            if all(
                str(img) in progress_data
                and os.path.exists(os.path.join(self.output_folder, f"pyrImg{img}.mat"))
                for img in range(start_idx, end_idx)
            ):
                print(f"Skipping batch {batch_idx} (already processed)")
                continue

            self.process_batch(start_idx, end_idx)

            # Save progress to log file after each batch
            with open(log_file, "w") as f:
                json.dump(progress_data, f, indent=4)

        print(
            f"Finished processing all images in {time.time() - total_start_time:.2f} sec."
        )


def visualize_image_with_check(processor, img_num):
    """Visualize and check dimensions of the processed image."""
    img = processor.load_image(img_num)
    print(f"Original Image Shape: {img.shape}")

    # Interpolation
    img_interpolated = processor.interpolate_image(img)
    print(f"Interpolated Image Shape: {img_interpolated.shape}")

    # Add fixation point
    img_with_fixation = processor.add_fixation_point(img_interpolated)

    # Pad with gray background
    img_padded = processor.pad_background(img_with_fixation)
    print(f"Padded Image Shape: {img_padded.shape}")

    # Grayscale Conversion
    img_gray = np.mean(img_padded, axis=2)
    print(f"Grayscale Image Shape: {img_gray.shape}")

    # Downsample to 512x512
    img_downsampled = cv2.resize(
        img_gray, (processor.dims[1], processor.dims[0]), interpolation=cv2.INTER_AREA
    )
    print(f"Downsampled Image Shape: {img_downsampled.shape}")

    # Display
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(img_with_fixation / 255.0)  # Normalize for display
    axs[0].set_title("With Fixation")
    axs[0].axis("off")

    axs[1].imshow(img_padded / 255.0)  # Normalize for display
    axs[1].set_title("Padded Image")
    axs[1].axis("off")

    axs[2].imshow(img_downsampled, cmap="gray")
    axs[2].set_title("Downsampled Image")
    axs[2].axis("off")

    plt.show()


if __name__ == "__main__":
    # Initialize the processor for all 73k images
    output_folder = "D:/NSD/pyramid_expand/"  # Change to your preferred path
    processor = NSDStimulusProcessor(
        hdf5_file, output_folder=output_folder, max_images=73000
    )

    # Process all stimuli in batches of 10k
    processor.process_all_in_batches(batch_size=73000)
    # processor.process_image(0)  # Replace 3 with any image number
    # visualize_image_with_check(processor, img_num=3)
    # processor.visualize_pyramid(3)


"""
If an error occurs (e.g., session crashes after processing 5,000 out of 10,000 images in a batch):

Re-run the batch:
processor.process_batch(start_idx=0, end_idx=10000)
The code will skip the first 5,000 images (already saved) and resume processing from image 5,001.
"""

In [ ]:
import scipy.io as sio
import matplotlib.pyplot as plt
import numpy as np

# Load the .mat file
mat_file_path = "/content/drive/My Drive/Uni/msc/Zvi Lab/nsd/V1_Drift/pyrImg2.mat"
data = sio.loadmat(mat_file_path)

# Print available keys (variables inside the .mat file)
print("🔍 Keys in the .mat file:", data.keys())

# Inspect an example variable
print("🔹 interpImgSize:", data["interpImgSize"])
print("🔹 bigImg shape:", data["bigImg"].shape)
print("🔹 sumOri shape:", len(data["sumOri"]))
print("🔹 modelOri shape:", len(data["modelOri"]))


# Load the .mat file
data = sio.loadmat(mat_file_path)

# Extract and visualize the grayscale image
bigImg = data["bigImg"]

plt.figure(figsize=(6, 6))
plt.imshow(bigImg, cmap="gray")
plt.title("Loaded Image from pyrImg3.mat")
plt.axis("off")
plt.show()

# Extract and visualize the first orientation of the first level
sumOri = data["sumOri"]
modelOri = data["modelOri"]

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 2)
plt.imshow(
    np.array(modelOri[0][0][0]), cmap="gray"
)  # Choose the first orientation at the first level
plt.title("First Orientation at Level 1")
plt.axis("off")
plt.show()